In [2]:
from pathlib import Path
import json
import re
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd


# project directory
project_directory = Path('../processing/results')

In [6]:
def strip_html(cell: str) -> str:
    # collapse <sub>x</sub> into a plain suffix (e.g. "R<sub>merge</sub>" -> "R_merge")
    cell = re.sub(r"<sub>(.*?)</sub>", r"_\1", cell)
    # drop any remaining tags (<strong>, </strong>, etc.)
    return re.sub(r"<[^>]+>", "", cell).strip()

def extract_summary_table(scaling_report):
    summary_table = scaling_report['scaling_tables']['overall']
    rows = [[strip_html(cell) for cell in r] for r in summary_table]
    header, *body = rows
    df = pd.DataFrame(body, columns=header).set_index(header[0])
    return df

def extract_frame_range(scaling_report):
    return scaling_report['image_range_tables'][0][1][1]

rows = []
for p in project_directory.glob("**/dials/truncated/scaled_truncated.json"):
    synchrotron, date_str = p.parts[-5].split('_')
    date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    row = {"synchrotron":synchrotron.upper(), "date":date, "crystal":p.parts[-4]}
    with open(p,'r') as f:
        scaling_report = json.load(f)
    summary_table = extract_summary_table(scaling_report)
    if row['crystal'].endswith('merge'):
        row['crystal'] = 'merged'
        row['Frames'] = ''
    else:
        frame_range = extract_frame_range(scaling_report)
        #degrees = (frame_range[1]-frame_range[0])/10
        #row['frame range'] = f'{frame_range[0]}-{frame_range[1]} ({round(degrees)}˚)'
        row['Frames'] = f'{frame_range[0]}-{frame_range[1]}'
    resol = summary_table['High resolution']['Resolution (Å)']
    row['CC_½'] = summary_table['High resolution']['CC_½']
    row['Completeness'] = summary_table['High resolution']['Completeness']
    rows.append(row)
    
df_scaling = pd.DataFrame(rows).set_index(["synchrotron", "date", "crystal"]).sort_index()
df_scaling

Frames   CC_½ Completeness
synchrotron date       crystal                            
ALS         2025-10-15 mac1_10  1-2800  0.871       99.98%
                       mac1_12  1-2800  0.942      100.00%
                       mac1_13  1-2800  0.929      100.00%
                       merged           0.980      100.00%
            2025-11-25 mac1_2   1-3600  0.863       93.99%
                       mac1_4   1-3600  0.891       99.08%
                       merged           0.939      100.00%
            2026-04-23 mac1_4   1-3600  0.805      100.00%
                       mac1_6   1-3600  0.606      100.00%
                       mac1_7   1-3600  0.821      100.00%
CHESS       2025-09-24 mac1_2   1-3600  0.921      100.00%
                       mac1_3   1-3600  0.940      100.00%
                       mac1_5   1-3600  0.944      100.00%
                       merged           0.981      100.00%
            2025-11-05 mac1_4   1-7200  0.988      100.00%
                       mac1_5   1-7200  0.984      100.00%
                       merged           0.994      100.00%

In [8]:
rows = []
for p in project_directory.glob("**/dials/estimate_resolution/dials.estimate_resolution.log"):
    t = p.read_text(errors="ignore")
    synchrotron, date_str = p.parts[-5].split('_')
    date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    row = {"synchrotron":synchrotron.upper(), "date":date, "crystal":p.parts[-4]}
    m = re.search(r'Resolution cc_half:\s*([0-9.]+)', t, re.M)
    row['Resolution'] = m.group(1) if m else None
    rows.append(row)
df_resol = pd.DataFrame(rows).set_index(["synchrotron", "date", "crystal"]).sort_index()
df_resol

Resolution
synchrotron date       crystal           
ALS         2025-10-15 mac1_10       1.08
                       mac1_12       1.05
                       mac1_13       1.06
            2025-11-25 mac1_2        1.12
                       mac1_4        1.11
            2026-04-23 mac1_4        1.09
                       mac1_6        1.16
                       mac1_7        1.09
CHESS       2025-09-24 mac1_2        1.05
                       mac1_3        1.02
                       mac1_5        1.02
            2025-11-05 mac1_4        1.02
                       mac1_5        1.02

In [9]:
# integration stats (mosaicity, cell params)
pats = {
    "Mosaicity": r"sigma m:\s*([0-9.]+)",
    "Unit Cell": r"Unit cell:\s*(.+)",
}
rows = []
for p in project_directory.glob("**/dials/dials.integrate.log"):
    t = p.read_text(errors="ignore")
    synchrotron, date_str = p.parts[-4].split('_')
    date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    row = {"synchrotron":synchrotron.upper(), "date":date, "crystal":p.parts[-3]}
    for k, pat in pats.items():
        m = re.search(pat, t, re.M)
        row[k] = m.group(1) if m else None
    rows.append(row)
    # fix the unit cell cloumn
    if row["Unit Cell"] is not None:
        row["Unit Cell"] = tuple(
            float(re.sub(r"\(.*?\)", "", x).strip())
            for x in row["Unit Cell"].split(",")
        )
df_integrate = pd.DataFrame(rows).set_index(["synchrotron", "date", "crystal"]).sort_index()
df_integrate['a'] = df_integrate['Unit Cell'].apply(lambda t: round(t[0],2))
df_integrate['c'] = df_integrate['Unit Cell'].apply(lambda t: round(t[2],2))
df_integrate['Mosaicity'] = pd.to_numeric(df_integrate['Mosaicity']).round(3)
#df_integrate = df_integrate.rename(columns={'Mosaicity': 'Mosaicity (˚)'})
df_integrate = df_integrate.drop(columns=['Unit Cell'])
df_integrate

Mosaicity      a      c
synchrotron date       crystal                         
ALS         2025-07-01 xtal10       0.186  89.17  39.65
                       xtal3        0.120  89.03  39.47
                       xtal7        0.023  89.00  39.70
                       xtal9        0.009  89.23  40.13
            2025-10-15 mac1_10      0.014  89.27  40.30
                       mac1_12      0.014  89.24  40.22
                       mac1_13      0.014  89.24  40.20
                       mac1_7       0.056  89.25  40.19
            2025-11-25 mac1_2       0.014  89.22  40.16
                       mac1_4       0.009  89.23  40.20
            2026-04-23 mac1_4       0.012  89.24  40.30
                       mac1_6       0.051  89.16  40.08
                       mac1_7       0.029  89.16  40.05
CHESS       2025-09-24 mac1_2       0.004  88.41  39.91
                       mac1_3       0.004  88.42  39.94
                       mac1_5       0.003  88.41  39.89
            2025-11-05 mac1_4       0.009  88.42  39.95
                       mac1_5       0.008  88.44  39.97

In [10]:
df = df_scaling.join(df_resol).join(df_integrate)
df = df[['Frames','Mosaicity','Resolution','a','c','CC_½','Completeness']]
df.columns = pd.MultiIndex.from_tuples([
     ('Frames', ''),                 # Flat column (has a blank placeholder level)
     ('Mosaicity', ''),
     ('Resolution', ''),
     ('Unit Cell', 'a'),
     ('Unit Cell', 'b'),
     ('Outer shell (1.27-1.25 Å)', 'CC_½'),       # Nested column
     ('Outer shell (1.27-1.25 Å)', 'Completeness')        # Nested column
])
#df.style.format(na_rep="")
df

Frames Mosaicity Resolution Unit Cell         \
                                                                    a      b   
synchrotron date       crystal                                                 
ALS         2025-10-15 mac1_10  1-2800     0.014       1.08     89.27  40.30   
                       mac1_12  1-2800     0.014       1.05     89.24  40.22   
                       mac1_13  1-2800     0.014       1.06     89.24  40.20   
                       merged                NaN        NaN       NaN    NaN   
            2025-11-25 mac1_2   1-3600     0.014       1.12     89.22  40.16   
                       mac1_4   1-3600     0.009       1.11     89.23  40.20   
                       merged                NaN        NaN       NaN    NaN   
            2026-04-23 mac1_4   1-3600     0.012       1.09     89.24  40.30   
                       mac1_6   1-3600     0.051       1.16     89.16  40.08   
                       mac1_7   1-3600     0.029       1.09     89.16  40.05   
CHESS       2025-09-24 mac1_2   1-3600     0.004       1.05     88.41  39.91   
                       mac1_3   1-3600     0.004       1.02     88.42  39.94   
                       mac1_5   1-3600     0.003       1.02     88.41  39.89   
                       merged                NaN        NaN       NaN    NaN   
            2025-11-05 mac1_4   1-7200     0.009       1.02     88.42  39.95   
                       mac1_5   1-7200     0.008       1.02     88.44  39.97   
                       merged                NaN        NaN       NaN    NaN   

                               Outer shell (1.27-1.25 Å)               
                                                    CC_½ Completeness  
synchrotron date       crystal                                         
ALS         2025-10-15 mac1_10                     0.871       99.98%  
                       mac1_12                     0.942      100.00%  
                       mac1_13                     0.929      100.00%  
                       merged                      0.980      100.00%  
            2025-11-25 mac1_2                      0.863       93.99%  
                       mac1_4                      0.891       99.08%  
                       merged                      0.939      100.00%  
            2026-04-23 mac1_4                      0.805      100.00%  
                       mac1_6                      0.606      100.00%  
                       mac1_7                      0.821      100.00%  
CHESS       2025-09-24 mac1_2                      0.921      100.00%  
                       mac1_3                      0.940      100.00%  
                       mac1_5                      0.944      100.00%  
                       merged                      0.981      100.00%  
            2025-11-05 mac1_4                      0.988      100.00%  
                       mac1_5                      0.984      100.00%  
                       merged                      0.994      100.00%

In [11]:
# export
df.to_csv("../figures/source_data/dials_stats.csv")

## Humidity tests

In [12]:
# merge stats (Wilson B)
pats = {
    "Wilson B": r"Wilson B factor\s*([0-9.]+)",
}
rows = []
for p in project_directory.glob("**/dials/dials.merge.log"):
    t = p.read_text(errors="ignore")
    synchrotron, date_str = p.parts[-4].split('_')
    date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    row = {"synchrotron":synchrotron.upper(), "date":date, "crystal":p.parts[-3]}
    for k, pat in pats.items():
        m = re.search(pat, t, re.M)
        row[k] = m.group(1) if m else None
    rows.append(row)

df_merge = pd.DataFrame(rows).set_index(["synchrotron", "date", "crystal"]).sort_index()
df_merge

Wilson B
synchrotron date       crystal            
ALS         2025-07-01 xtal10       13.175
                       xtal3        17.296
                       xtal7        14.175
                       xtal9        12.616
            2025-10-15 mac1_7       13.912
                       mac1_merge   13.191
            2025-11-25 mac1_merge   12.946
            2026-04-23 mac1_merge   12.438
CHESS       2025-09-24 mac1_merge   13.182
            2025-11-05 mac1_merge   12.944

In [13]:
df = df_integrate.join(df_merge,how='outer')
df = df.loc['ALS'] # drop the CHESS data 
df = df.drop([('2025-10-15','mac1_7')], axis=0) # drop the mac1_7 dataset (radiation damage test)
df = df.drop('2026-04-23',axis=0) # drop the small crystal dataset
df

Mosaicity      a      c Wilson B
date       crystal                                     
2025-07-01 xtal10          0.186  89.17  39.65   13.175
           xtal3           0.120  89.03  39.47   17.296
           xtal7           0.023  89.00  39.70   14.175
           xtal9           0.009  89.23  40.13   12.616
2025-10-15 mac1_10         0.014  89.27  40.30      NaN
           mac1_12         0.014  89.24  40.22      NaN
           mac1_13         0.014  89.24  40.20      NaN
           mac1_merge        NaN    NaN    NaN   13.191
2025-11-25 mac1_2          0.014  89.22  40.16      NaN
           mac1_4          0.009  89.23  40.20      NaN
           mac1_merge        NaN    NaN    NaN   12.946

In [14]:
df.to_csv("../figures/source_data/dials_stats_humidity.csv")